# Customer Churn Prediction Pipeline

Predict which customers are likely to churn based on their usage patterns.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

In [ ]:
# Generate synthetic customer data
np.random.seed(42)
n_customers = 1000

data = {
    'customer_id': range(1, n_customers + 1),
    'account_length': np.random.randint(1, 200, n_customers),
    'total_charges': np.random.uniform(50, 5000, n_customers),
    'monthly_charges': np.random.uniform(20, 150, n_customers),
    'num_products': np.random.randint(1, 5, n_customers),
    'support_calls': np.random.randint(0, 10, n_customers),
    'contract_type': np.random.choice(['monthly', 'annual', 'biannual'], n_customers),
}

df = pd.DataFrame(data)

In [ ]:
# Create churn target based on features (customers with high support calls and short contracts more likely to churn)
churn_probability = (
    0.3 * (df['support_calls'] > 5).astype(int) +
    0.2 * (df['contract_type'] == 'monthly').astype(int) +
    0.2 * (df['account_length'] < 50).astype(int) +
    0.15 * (df['monthly_charges'] > 100).astype(int) +
    0.15 * (df['num_products'] == 1).astype(int)
)

df['churned'] = (churn_probability + np.random.uniform(-0.2, 0.2, n_customers) > 0.5).astype(int)

In [ ]:
# Data cleaning and preprocessing
df_clean = df.copy()
df_clean['avg_monthly_charge'] = df_clean['total_charges'] / df_clean['account_length']
df_clean['calls_per_month'] = df_clean['support_calls'] / (df_clean['account_length'] / 30)
df_clean = pd.get_dummies(df_clean, columns=['contract_type'], drop_first=True)

print(f"Dataset shape: {df_clean.shape}")
print(f"Churn rate: {df_clean['churned'].mean():.2%}")

In [ ]:
# Prepare features and target
feature_columns = [
    'account_length', 'total_charges', 'monthly_charges',
    'num_products', 'support_calls', 'avg_monthly_charge',
    'calls_per_month', 'contract_type_biannual', 'contract_type_monthly'
]

X = df_clean[feature_columns]
y = df_clean['churned']

In [ ]:
# Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set size: {len(X_train)}")
print(f"Test set size: {len(X_test)}")

In [ ]:
# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Train Random Forest model
model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

print("Training model...")
model.fit(X_train_scaled, y_train)
print("Model training complete!")

In [ ]:
# Make predictions
y_train_pred = model.predict(X_train_scaled)
y_test_pred = model.predict(X_test_scaled)
y_test_proba = model.predict_proba(X_test_scaled)[:, 1]

In [ ]:
# Calculate metrics
train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy = accuracy_score(y_test, y_test_pred)
test_precision = precision_score(y_test, y_test_pred)
test_recall = recall_score(y_test, y_test_pred)
test_f1 = f1_score(y_test, y_test_pred)

print("\n" + "="*50)
print("MODEL PERFORMANCE METRICS")
print("="*50)
print(f"\nTraining Accuracy:   {train_accuracy:.4f}")
print(f"Test Accuracy:       {test_accuracy:.4f}")
print(f"\nPrecision:          {test_precision:.4f}")
print(f"Recall:             {test_recall:.4f}")
print(f"F1 Score:           {test_f1:.4f}")
print("\n" + "="*50)

In [ ]:
# Show confusion matrix
cm = confusion_matrix(y_test, y_test_pred)
print("\nConfusion Matrix:")
print(f"                Predicted")
print(f"              No Churn  Churned")
print(f"Actual No      {cm[0,0]:6d}   {cm[0,1]:6d}")
print(f"       Churn   {cm[1,0]:6d}   {cm[1,1]:6d}")

In [ ]:
# Feature importance analysis
feature_importance = pd.DataFrame({
    'feature': feature_columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nTop 5 Most Important Features:")
print("="*50)
for idx, row in feature_importance.head(5).iterrows():
    print(f"{row['feature']:25s} {row['importance']:.4f}")

In [ ]:
# Predict on new customers
print("\n" + "="*50)
print("PREDICTIONS ON HIGH-RISK CUSTOMERS")
print("="*50)

# Find customers with high churn probability
high_risk_idx = np.where(y_test_proba > 0.7)[0]
high_risk_customers = X_test.iloc[high_risk_idx]

print(f"\nFound {len(high_risk_customers)} high-risk customers (>70% churn probability)")
print("\nSample high-risk customer profiles:")
for i, (idx, customer) in enumerate(high_risk_customers.head(3).iterrows()):
    prob = y_test_proba[high_risk_idx[i]]
    print(f"\nCustomer {idx}:")
    print(f"  Churn Probability: {prob:.2%}")
    print(f"  Account Length: {customer['account_length']:.0f} days")
    print(f"  Support Calls: {customer['support_calls']:.0f}")
    print(f"  Monthly Charges: ${customer['monthly_charges']:.2f}")

In [ ]:
# Summary statistics
print("\n" + "="*50)
print("SUMMARY")
print("="*50)
print(f"\nTotal customers analyzed: {len(df)}")
print(f"Actual churn rate: {y_test.mean():.2%}")
print(f"Predicted churn rate: {y_test_pred.mean():.2%}")
print(f"\nModel correctly identified {test_recall:.2%} of actual churners")
print(f"Of predicted churners, {test_precision:.2%} actually churned")
print(f"\nHigh-risk customers to follow up with: {len(high_risk_customers)}")